
# Aphasia Treatment Analysis: Evaluating the BEARS Framework

## 1. Study Context & Conceptual Corrections

This notebook provides a research-grade analysis of aphasia treatment data, moving from descriptive visualizations to inferential statistical modeling. The analysis centers on evaluating naming treatment performance, semantic feature verification, and longitudinal recovery trajectories.

### The BEARS Framework: Explicit Definition
**BEARS** stands for **Balancing Effort, Accuracy, and Response Speed** (Evans et al., 2021). 

**What BEARS is:**
* A **treatment design framework** aimed at optimizing retrieval practice.
* A method to manage **speed–accuracy tradeoffs** to maximize learning efficiency.
* A focus on **system calibration**, helping persons with aphasia (PWA) match task effort to their current language capabilities.

**What BEARS is NOT:**
* A **Memory Decay Model** (e.g., Ebbinghaus).
* **Bayesian Knowledge Tracing (BKT)** or **Half-Life Regression (HLR)**, which focus on mastery probability and spacing.
* A **Leitner System** or simple flashcard repetition algorithm.

### Semantic Feature Verification (SFV) vs. Semantic Feature Analysis (SFA)
**SFV** is a modification of **SFA**. While SFA involves *generating* features to support lexical retrieval, SFV involves *verifying* provided features. 
* SFV is **not a free naming task**.
* **Retrieval practice** and **feature verification** are separate mechanisms. 
* Evidence (Cavanaugh et al., 2022) suggests that successful retrieval practice—not feature verification practice itself—is the primary predictor of naming outcomes for treated words.

### Notebook Objectives
1. **Statistically corrected analysis**: Implementing regression-based modeling to control for baseline severity.
2. **Speed-Accuracy Tradeoff**: Incorporating Response Time (RT) analysis as a core clinical metric, filtered strictly for correct trials.
3. **Research-grade interpretation**: Avoiding causal overclaims and addressing aggregation bias.



## 2. Dataset Overview & Data Loading

### Variable Definitions
* **baseline**: Initial probe accuracy measured before intervention.
* **treatment**: Performance metrics collected during the active intervention phase.
* **followup**: Accuracy measured at the maintenance interval post-treatment.
* **improvement**: Defined here as `followup - baseline`. (Note: In modeling, we control for baseline explicitly).
* **score**: Standardized clinical measures (e.g., WAB, PNT).
* **MPO**: Months Post-Onset (Time since brain injury).
* **education / age**: Standard demographic covariates.

### Handling Aggregation Bias
Analyzing data at the **session level** (e.g., `df.groupby('session').mean()`) provides a cohort-level learning curve but hides critical **trial-level dynamics**. This aggregation destroys the distribution of Response Times (essential for Ex-Gaussian modeling) and erases item-specific variance. While this notebook presents session-level summaries for visualization, inferential results transition to regression models where possible.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (12, 6)

# Load core datasets and clean index columns
files = {
    "probes": "1. 2019-11-12_naming_probes_tidy.csv",
    "retrieval_noprime": "2. 2019-10-25_treatment_retrieval_noprime.csv",
    "feature_ver": "3. 2019-10-25_treatment_feature_ver.csv",
    "retrieval_prime": "4. 2019-10-25_treatment_retrieval_prime.csv",
    "gamification": "5. 2019-12-3_coins-stars.csv",
    "outcomes": "6. 2019-10-3_Outcome_data_tidy.csv"
}

dfs = {}
for name, filename in files.items():
    if os.path.exists(filename):
        df = pd.read_csv(filename)
        # Remove unnamed columns often found in exported CSVs
        df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
        
        if 'player' in df.columns:
            df['player'] = df['player'].str.lower()
        elif 'Player' in df.columns:
             df['player'] = df['Player'].str.lower()
             df.rename(columns={'Player': 'player'}, inplace=True)
        dfs[name] = df
    else:
        print(f"File not found: {filename}")

print(f"Loaded {len(dfs)} datasets. Variable mapping complete.")



## 3. Treatment Performance Analysis (Accuracy & Speed)

### Evaluating the Speed-Accuracy Tradeoff
According to BEARS principles, **accuracy is not the sole metric of recovery**. Recovery in the BEARS framework is defined by the **co-optimization of speed and accuracy**.

**Methodological Note on RT Analysis**: 
Response Time (RT) is analyzed strictly for **Correct Trials**. Including error-trial RTs would contaminate the data with different cognitive processes (e.g., frustration, search-withdrawal). Decreasing RT on correct trials indicates improved **retrieval efficiency**.


In [ ]:

def process_treatment(df, acc_col, rt_col, label):
    # 1. Accuracy aggregation (All trials)
    acc_sess = df.groupby(['session', 'player'])[acc_col].mean().reset_index()
    
    # 2. RT aggregation (Correct trials only - BEARS Best Practice)
    rt_sess = df[df[acc_col] == 1].groupby(['session', 'player'])[rt_col].mean().reset_index()
    
    # Merge session-level accuracy and RT
    p_sess = pd.merge(acc_sess, rt_sess, on=['session', 'player'], how='outer')
    
    # 3. Aggregating to cohort level
    agg = p_sess.groupby('session')[[acc_col, rt_col]].agg(['mean', 'std', 'count']).reset_index()
    
    # Explicitly flatten columns or handle manually to avoid index bugs
    # We will compute SE and CI manually for each metric
    for metric in [acc_col, rt_col]:
        mn = agg[(metric, 'mean')]
        std = agg[(metric, 'std')]
        cnt = agg[(metric, 'count')]
        se = std / np.sqrt(cnt)
        agg[(metric, 'se')] = se
        agg[(metric, 'ci_up')] = mn + 1.96 * se
        agg[(metric, 'ci_low')] = mn - 1.96 * se
    
    agg['Task'] = label
    return agg, p_sess

if all(k in dfs for k in ['retrieval_noprime', 'retrieval_prime', 'feature_ver']):
    # Process each task
    agg_np, sum_np = process_treatment(dfs['retrieval_noprime'], 'naming1_resp.corr', 'naming1_vocal.rt', 'Retrieval (No Prime)')
    agg_p, sum_p = process_treatment(dfs['retrieval_prime'], 'naming2_resp.corr', 'naming2_vocal.rt', 'Retrieval (Prime)')
    agg_fv, sum_fv = process_treatment(dfs['feature_ver'], 'sf_resp.corr', 'sf_resp.rt', 'Feature Verification')

    # Visualizing Accuracy Trends
    plt.figure(figsize=(15, 7))
    plt.subplot(1, 2, 1)
    for a, l in zip([agg_np, agg_p, agg_fv], ['No Prime', 'Prime', 'Feature Ver']):
        # Plotting Mean Accuracy
        plt.plot(a['session'], a[(a.columns[1][0], 'mean')], label=l, linewidth=2.5)
        # Shading 95% CI
        plt.fill_between(a['session'], 
                         a[(a.columns[1][0], 'ci_low')], 
                         a[(a.columns[1][0], 'ci_up')], 
                         alpha=0.15)
    plt.title('Mean Accuracy Progression')
    plt.ylabel('Probability Correct')
    plt.xlabel('Session Index')
    plt.legend()

    # Visualizing Response Time (RT) Trends
    plt.subplot(1, 2, 2)
    for a, l in zip([agg_np, agg_p, agg_fv], ['No Prime', 'Prime', 'Feature Ver']):
        # Plotting Mean Correct RT
        plt.plot(a['session'], a[(a.columns[4][0], 'mean')], label=f"{l} (Correct RT)", linestyle='--', linewidth=2)
    plt.title('Mean Correct Response Time (RT) Progression')
    plt.ylabel('RT (Seconds)')
    plt.xlabel('Session Index')
    plt.legend()
    plt.tight_layout()
    plt.show()

    print("Note: Shaded regions represent 95% Confidence Intervals for accuracy.")


### Trial-Level Dynamics: Accuracy Progression per Word Encounter

While session-level aggregation reveals broad learning curves, it can mask the specific rate of improvement per item. In the BEARS framework, we are interested in how many encounters it takes for a word to reach mastery. The following scatter plot shows the **Mean Accuracy** as a function of the **Trial Index** (the n-th time a specific player has encountered a specific word).

In [ ]:
def process_trial_level(df, acc_col, label):
    # Calculate cumulative encounter count for each word per player
    temp_df = df.copy()
    # Sort by session and then index to ensure temporal order if not already sorted
    temp_df = temp_df.sort_values(['player', 'session'])
    temp_df['trial_index'] = temp_df.groupby(['player', 'stim_text']).cumcount() + 1
    
    # Aggregate accuracy by trial index
    agg = temp_df.groupby('trial_index')[acc_col].agg(['mean', 'count']).reset_index()
    
    # Filter to indices with at least 5 observations to reduce noise at higher encounter counts
    agg = agg[agg['count'] >= 5]
    return agg

if all(k in dfs for k in ['retrieval_noprime', 'retrieval_prime', 'feature_ver']):
    plt.figure(figsize=(12, 7))
    
    tasks = [
        (dfs['retrieval_noprime'], 'naming1_resp.corr', 'No Prime', 'o'),
        (dfs['retrieval_prime'], 'naming2_resp.corr', 'Prime', 's'),
        (dfs['feature_ver'], 'sf_resp.corr', 'Feature Ver', '^')
    ]
    
    for df, acc_col, label, marker in tasks:
        trial_data = process_trial_level(df, acc_col, label)
        plt.scatter(trial_data['trial_index'], trial_data['mean'], label=label, marker=marker, alpha=0.7, s=60)
        
        # Add a simple moving average or trendline for better visibility
        if len(trial_data) > 1:
            z = np.polyfit(trial_data['trial_index'], trial_data['mean'], 2)
            p = np.poly1d(z)
            x_new = np.linspace(trial_data['trial_index'].min(), trial_data['trial_index'].max(), 100)
            plt.plot(x_new, p(x_new), "--", alpha=0.5)

    plt.title('Mean Accuracy Progression per Trial (Word Encounters)')
    plt.ylabel('Mean Accuracy (Probability Correct)')
    plt.xlabel('Cumulative Trial Index (n-th encounter of word)')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

### Individual Player Focus: Word-Level Progression (CDA001)

This section drills down into the performance of the first player (**CDA001**) to visualize how individual words progress over multiple encounters. We focus on the **No Prime** and **Prime** conditions to compare learning rates at the individual level.

In [ ]:
# Individual Player Analysis for CDA001
first_player = dfs['retrieval_noprime']['player'].unique()[0]

def plot_word_progression_individual(player_id):
    plt.figure(figsize=(12, 7))
    
    tasks = [
        (dfs['retrieval_noprime'], 'naming1_resp.corr', 'No Prime', 'o', '#1f77b4'),
        (dfs['retrieval_prime'], 'naming2_resp.corr', 'Prime', 's', '#ff7f0e')
    ]
    
    for df, acc_col, label, marker, color in tasks:
        # Filter for specific player
        p_df = df[df['player'] == player_id].copy()
        
        # Calculate trial index for each word
        p_df = p_df.sort_values('session')
        p_df['trial_index'] = p_df.groupby('stim_text').cumcount() + 1
        
        # Aggregate mean across words for each encounter number
        player_agg = p_df.groupby('trial_index')[acc_col].mean().reset_index()
        
        # Plot the mean progression as a scatter plot
        plt.scatter(player_agg['trial_index'], player_agg[acc_col], 
                    label=f"{label} (Player Mean)", marker=marker, color=color, s=100, edgecolors='white', zorder=10)
        
        # Add individual word trials as smaller, jittered background points to see the distribution
        # We'll use a small jitter on Y to separate the 0s and 1s
        jitter = np.random.uniform(-0.03, 0.03, size=len(p_df))
        plt.scatter(p_df['trial_index'], p_df[acc_col] + jitter, 
                    color=color, alpha=0.1, s=15, marker='.', label='_nolegend_')
        
        # Add trendline
        if len(player_agg) > 1:
            z = np.polyfit(player_agg['trial_index'], player_agg[acc_col], 2)
            p = np.poly1d(z)
            x_new = np.linspace(player_agg['trial_index'].min(), player_agg['trial_index'].max(), 100)
            plt.plot(x_new, p(x_new), '--', color=color, alpha=0.5, label=f'{label} Trend')

    plt.title(f'Individual Word Progression for Player {player_id}')
    plt.ylabel('Accuracy (individual trials vs mean)')
    plt.xlabel('Cumulative Trial Index (n-th encounter of word)')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig("cda001_mean_progression.png", dpi=300, bbox_inches="tight")
    plt.show()

plot_word_progression_individual(first_player)

### Enhanced Word Learning Comparison (Prime vs No-Prime)

This section evaluates how clinical priming affects word learning across different difficulty levels. 
Lines use the **`.-` style** (points connected by thin lines) to show the chronological trial sequence for player **CDA001**.

1. **Difficult**: Baseline accuracy near 0. Helps identify if priming breaks through persistent errors.
2. **Medium**: Partial mastery. Shows if priming accelerates the learning curve.
3. **Easy**: Mastery maintenance. Confirms consistent performance across both task types.

*Note: Figures are automatically saved as PNGs for report inclusion.*

In [ ]:
def plot_word_grid_comparison(player_id, word_list, title, filename_slug):
    p_df_np = dfs['retrieval_noprime'][dfs['retrieval_noprime']['player'] == player_id].copy()
    p_df_p = dfs['retrieval_prime'][dfs['retrieval_prime']['player'] == player_id].copy()
    
    n_words = len(word_list)
    rows = (n_words + 4) // 5
    fig, axes = plt.subplots(rows, 5, figsize=(24, 5 * rows), sharey=True)
    axes = axes.flatten()
    
    for i, word in enumerate(word_list):
        ax = axes[i]
        # Get No-Prime data
        word_np = p_df_np[p_df_np['stim_text'] == word].sort_values('session').copy()
        word_np['trial_index'] = range(1, len(word_np) + 1)
        
        # Get Prime data
        word_p = p_df_p[p_df_p['stim_text'] == word].sort_values('session').copy()
        word_p['trial_index'] = range(1, len(word_p) + 1)
        
        # Plot No-Prime (connected points style)
        ax.plot(word_np['trial_index'], word_np['naming1_resp.corr'], '.-', 
                color='#1f77b4', alpha=0.6, markersize=8, label='No-Prime')
        
        # Plot Prime (connected points style)
        ax.plot(word_p['trial_index'], word_p['naming1_resp.corr'], '.-', 
                color='#ff7f0e', alpha=0.6, markersize=8, label='Prime')
        
        ax.set_title(f"{word}", fontsize=15, fontweight='bold')
        ax.set_ylim(-0.1, 1.1)
        ax.set_xlabel("Trial #", fontsize=10)
        if i % 5 == 0: ax.set_ylabel("Accuracy", fontsize=10)
        ax.grid(True, linestyle=':', alpha=0.5)
        
    for j in range(i + 1, len(axes)): fig.delaxes(axes[j])
    
    # Create a single legend for the whole figure
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper right', bbox_to_anchor=(0.95, 0.98), fontsize=14)
    
    plt.suptitle(title, fontsize=24, y=1.03, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{filename_slug}.png", dpi=300, bbox_inches='tight')
    plt.show()

def analyze_player_comparison_tiers(player_id):
    # Calculate accuracy using No-Prime as baseline for difficulty tiers
    p_df = dfs['retrieval_noprime'][dfs['retrieval_noprime']['player'] == player_id].copy()
    word_stats = p_df.groupby('stim_text')['naming1_resp.corr'].mean().sort_values()
    
    difficult = word_stats.head(10).index.tolist()
    easy = word_stats.tail(10).index.tolist()[::-1]
    mid_idx = len(word_stats) // 2
    medium = word_stats.iloc[mid_idx-5:mid_idx+5].index.tolist()
    
    # Tier 1: Difficult
    plot_word_grid_comparison(player_id, difficult, 
                              f"Tier 1: Persistence Challenge - Difficult Words for Player {player_id}\n(Lowest baseline accuracy, comparing Prime vs No-Prime learning lines)", 
                              "cda001_tier1_difficult")
    
    # Tier 2: Medium
    plot_word_grid_comparison(player_id, medium, 
                              f"Tier 2: Learning Zone - Medium Words for Player {player_id}\n(Partial proficiency, comparing Prime vs No-Prime sequence)", 
                              "cda001_tier2_medium")
    
    # Tier 3: Easy
    plot_word_grid_comparison(player_id, easy, 
                              f"Tier 3: Mastery Check - Easy Words for Player {player_id}\n(High baseline accuracy, comparing Prime vs No-Prime consistency)", 
                              "cda001_tier3_easy")

analyze_player_comparison_tiers(first_player)

### Categorized Word Learning Trajectories (Player CDA001)

Learning dynamics often vary significantly depending on the word's baseline difficulty for the player. 
Below, we categorize words into three tiers:
1. **Difficult**: Words where the player had the lowest accuracy overall.
2. **Medium**: Words showing partial learning or inconsistent performance.
3. **Easy**: Words where the player achieved high accuracy consistently.

In [ ]:
def plot_word_grid_custom(player_id, word_list, title, filename_slug):
    # Extract data for the specific player from both conditions
    df_np = dfs['retrieval_noprime'][dfs['retrieval_noprime']['player'] == player_id].copy()
    df_p = dfs['retrieval_prime'][dfs['retrieval_prime']['player'] == player_id].copy()
    
    n_words = len(word_list)
    rows = (n_words + 4) // 5
    fig, axes = plt.subplots(rows, 5, figsize=(22, 4.5 * rows), sharey=True)
    axes = axes.flatten()
    
    for i, word in enumerate(word_list):
        ax = axes[i]
        
        # No-Prime data setup
        w_np = df_np[df_np['stim_text'] == word].sort_values('session').copy()
        w_np['trial_index'] = range(1, len(w_np) + 1)
        
        # Prime data setup
        w_p = df_p[df_p['stim_text'] == word].sort_values('session').copy()
        w_p['trial_index'] = range(1, len(w_p) + 1)
        
        # Plot No-Prime (Blue)
        ax.plot(w_np['trial_index'], w_np['naming1_resp.corr'], 
                marker='o', linestyle='-', color='#1f77b4', alpha=0.5, label='No-Prime')
        
        # Plot Prime (Orange)
        ax.plot(w_p['trial_index'], w_p['naming2_resp.corr'], 
                marker='o', linestyle='-', color='#ff7f0e', alpha=0.5, label='Prime')
        
        ax.set_title(f"{word}", fontsize=14, fontweight='bold')
        ax.set_ylim(-0.1, 1.1)
        ax.set_xlabel("Trial # (Encounter Number)", fontsize=8)
        if i % 5 == 0: 
            ax.set_ylabel("Response Accuracy\n(0=Error, 1=Correct)", fontsize=8)
        ax.grid(True, linestyle=':', alpha=0.6)
        
    for j in range(i + 1, len(axes)): fig.delaxes(axes[j])
    
    # Get handles for legend from the first subplot
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper right', bbox_to_anchor=(0.98, 0.98), fontsize=12)
    
    plt.suptitle(title, fontsize=22, y=1.02, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{filename_slug}.png", dpi=300, bbox_inches='tight')
    plt.show()

def analyze_player_difficulty_tiers(player_id):
    # Base difficulty calculation using No-Prime as stable baseline
    p_df = dfs['retrieval_noprime'][dfs['retrieval_noprime']['player'] == player_id].copy()
    word_stats = p_df.groupby('stim_text')['naming1_resp.corr'].mean().sort_values()
    
    difficult = word_stats.head(10).index.tolist()
    easy = word_stats.tail(10).index.tolist()[::-1]
    mid_idx = len(word_stats) // 2
    medium = word_stats.iloc[mid_idx-5:mid_idx+5].index.tolist()
    
    # Tier 1: Difficult
    plot_word_grid_custom(player_id, difficult, 
                         f"Tier 1: Persistence Challenge - Difficult Words for {player_id}\n(Lowest baseline accuracy, comparing Prime vs No-Prime raw feedback)", 
                         f"{player_id}_tier1_difficult")
    
    # Tier 2: Medium
    plot_word_grid_custom(player_id, medium, 
                         f"Tier 2: Learning Zone - Medium Words for {player_id}\n(Partial proficiency, comparing Prime vs No-Prime progression)", 
                         f"{player_id}_tier2_medium")
    
    # Tier 3: Easy
    plot_word_grid_custom(player_id, easy, 
                         f"Tier 3: Mastery Check - Easy Words for {player_id}\n(Highest baseline accuracy, comparing Prime vs No-Prime consistency)", 
                         f"{player_id}_tier3_easy")

analyze_player_difficulty_tiers(first_player)

### Categorized Word Learning Trajectories (Player CDA001)

Learning dynamics often vary significantly depending on the word's baseline difficulty for the player. 
Below, we categorize words into three tiers:
1. **Difficult**: Words where the player had the lowest accuracy overall.
2. **Medium**: Words showing partial learning or inconsistent performance.
3. **Easy**: Words where the player achieved high accuracy consistently.


## 4. Gamification Analysis: Are Rewards Associated with Success?

### Visualizing Reward Impact
In this section, we explore the relationship between the number of **stars earned** and the **accuracy** achieved in the treatment task. 

It is important to remember that this is an **associative** analysis. While we hope that stars motivate the patient (leading to better performance), it is also true that patients who perform better naturally earn more stars. This visualization helps clinicians see if a patient is "responsive" to the rewards system. 


In [ ]:

if 'gamification' in dfs and 'retrieval_noprime' in dfs:
    df_stars = dfs['gamification']
    # sum_np contains player-session accuracy from Section 3
    merged = pd.merge(df_stars, sum_np, on=['player', 'session'], how='inner')
    
    if not merged.empty:
        plt.figure(figsize=(10, 6))
        # Visualizing the relationship between stars and accuracy
        sns.regplot(data=merged, x='stars', y='naming1_resp.corr', 
                    scatter_kws={'alpha':0.5}, line_kws={'color':'red'})
        
        plt.title('Relationship: Stars Earned vs. Task Accuracy')
        plt.xlabel('Stars (Reward)')
        plt.ylabel('Accuracy (Retrieval Correctness)')
        plt.show()
        
        # Simple Correlation
        corr = merged['stars'].corr(merged['naming1_resp.corr'])
        print(f"Overall Correlation between Rewards and Accuracy: {corr:.2f}")
        print("Note: A positive correlation suggests that the patient's performance and reward accumulation move together.")
    else:
        print("Insufficient data to link gamification and performance.")



## 5. Probe Analysis & Controlling for Baseline

### Addressing Regression to the Mean
In aphasia recovery research, "improvement" is often biased by the baseline score. To avoid this, we use **Residualized Gain** or **ANCOVA-style** modeling: we predict `followup` using `baseline` as a covariate.


In [ ]:

if 'probes' in dfs:
    df_p = dfs['probes']
    probes = df_p.groupby(['player', 'phase'])['trial_resp.corr.hand'].mean().unstack().reset_index()
    
    if 'baseline' in probes.columns and 'followup' in probes.columns:
        # Trajectories
        phases = [p for p in ['baseline', 'treatment', 'followup'] if p in probes.columns]
        plt.figure(figsize=(10, 6))
        for _, row in probes.iterrows():
            plt.plot(phases, [row[p] for p in phases], marker='o', alpha=0.5, label=row['player'])
        plt.title('Patient-Level Longitudinal Probe Trajectories')
        plt.ylabel('Mean Accuracy')
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.show()

        # OLS Model controlling for baseline severity
        res_m = smf.ols('followup ~ baseline', data=probes).fit()
        print("--- Regression: Follow-up Accuracy conditioned on Baseline ---")
        print(res_m.summary().tables[1])



## 6. Demonstration: Ex-Gaussian Reaction Time Modeling 

This section reproduces the core reaction-time modeling logic used in the **BEARS** framework. As discussed, accuracy alone often reaches a ceiling; modeling the underlying distribution of Response Times (RT) provides a more sensitive measurement of retrieval stability.

### The Ex-Gaussian Model
Reaction time distributions in aphasia are typically right-skewed. The Ex-Gaussian distribution models RT as:
**RT = Normal(μ, σ) + Exponential(τ)**

Where:
* **μ (Mu)**: Central processing/motor execution speed.
* **σ (Sigma)**: Variability in central processing.
* **τ (Tau)**: The exponential tail, capturing long-tail retrieval episodes (markers of instability or high cognitive effort).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import exponnorm, pearsonr

# ===============================
# 1️⃣ Ex-Gaussian Fit Function
# ===============================
def fit_exgaussian(rt_array):
    rt_array = np.array(rt_array)
    rt_array = rt_array[~np.isnan(rt_array)]

    if len(rt_array) < 40:
        return np.nan, np.nan, np.nan

    try:
        K, loc, scale = exponnorm.fit(rt_array)
        mu = loc
        sigma = scale
        tau = K * scale
        return mu, sigma, tau
    except:
        return np.nan, np.nan, np.nan


# ===============================
# 2️⃣ Data Preparation
# ===============================
df_rt = dfs['retrieval_noprime'].copy()
rt_col = 'naming1_vocal.rt'
acc_col = 'naming1_resp.corr'

# Filter RT (seconds)
df_rt = df_rt[(df_rt[rt_col] > 0.15) & (df_rt[rt_col] < 30)]

results_exg = []

for player, group in df_rt.groupby('player'):
    correct_trials = group[group[acc_col] == 1][rt_col]
    mu, sigma, tau = fit_exgaussian(correct_trials)

    results_exg.append({
        'player': player,
        'accuracy': group[acc_col].mean(),
        'mu': mu,
        'sigma': sigma,
        'tau': tau
    })

exgauss_df = pd.DataFrame(results_exg).dropna()


# ===============================
# 3️⃣ Statistical Relationship
# ===============================
if not exgauss_df.empty:

    r, p = pearsonr(exgauss_df['accuracy'], exgauss_df['tau'])
    print(f"\nCorrelation between Accuracy and Tau: r = {r:.3f}, p = {p:.4f}")

    # ===============================
    # 4️⃣ Clinical Classification
    # ===============================
    acc_thresh = exgauss_df['accuracy'].median()
    tau_thresh = exgauss_df['tau'].median()

    def classify(row):
        if row['accuracy'] < acc_thresh and row['tau'] > tau_thresh:
            return 'Unlearned'
        elif row['accuracy'] >= acc_thresh and row['tau'] > tau_thresh:
            return 'Fragile'
        elif row['accuracy'] >= acc_thresh and row['tau'] <= tau_thresh:
            return 'Stable'
        else:
            return 'Emerging'

    exgauss_df['state'] = exgauss_df.apply(classify, axis=1)

    # ===============================
    # 5️⃣ Clean Visualization
    # ===============================
    plt.figure(figsize=(10, 6))

    sns.scatterplot(
        data=exgauss_df,
        x='accuracy',
        y='tau',
        hue='state',
        palette={
            'Stable': 'green',
            'Fragile': 'orange',
            'Unlearned': 'red',
            'Emerging': 'blue'
        },
        s=180,
        edgecolor='black'
    )

    # Regression line
    coef = np.polyfit(exgauss_df['accuracy'], exgauss_df['tau'], 1)
    poly1d_fn = np.poly1d(coef)
    x_vals = np.linspace(exgauss_df['accuracy'].min(),
                         exgauss_df['accuracy'].max(), 100)
    plt.plot(x_vals, poly1d_fn(x_vals), color='black', linewidth=2)

    plt.axvline(acc_thresh, linestyle='--', alpha=0.5)
    plt.axhline(tau_thresh, linestyle='--', alpha=0.5)

    plt.xlabel('Mean Accuracy')
    plt.ylabel('Tau (Retrieval Instability)')
    plt.title(
        f'Accuracy vs Retrieval Instability (Ex-Gaussian)\n'
        f'r = {r:.2f}, p = {p:.3f}'
    )

    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    display(exgauss_df.sort_values('tau', ascending=False))

else:
    print("Insufficient data per player to fit stable Ex-Gaussian models.")